# 04 · Daten ausserhalb des IFC: Airtable und Tabellen

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LTplus-AG/bxmid03-hs26/blob/main/notebooks/2026-11-03_Validierung/04_Airtable_Daten_ausserhalb_IFC.ipynb)

Nicht jede Information muss **im** IFC stehen. Kosten, Fertigungsstatus oder Lieferanten werden oft in einer Tabelle gepflegt –
das IFC trägt nur einen **Link** dorthin. Der Schlüssel ist die **GlobalId**.

| Teil | Was ihr macht | Konto nötig? |
|---|---|---|
| A | Ein Modell mit Airtable-Links erkunden | nein |
| B | Aus einem IFC eine eigene Tabelle machen (CSV) | nein |
| C | Tabelle in Airtable oder Google Sheets bearbeiten | Airtable (gratis) oder Google |
| D | Tabelle zurück ans Modell hängen und prüfen | nein (Beispieldatei vorhanden) |

**Zuerst:** *Laufzeit → Alle ausführen* oder Zelle für Zelle mit ▶.

In [ ]:
#@title Vorbereitung (einmal ausführen)
!pip install -q ifcopenshell pandas
import os, urllib.request, pandas as pd, ifcopenshell, ifcopenshell.util.element as ue
BASIS = "https://raw.githubusercontent.com/LTplus-AG/bxmid03-hs26/main/modelle/"
def hole(datei):
    """Datei aus dem Kurs-Repository laden (falls noch nicht da)."""
    if not os.path.exists(datei):
        urllib.request.urlretrieve(BASIS + datei, datei)
    return datei
try:
    from google.colab import files      # nur in Colab vorhanden
except ImportError:
    files = None
print("bereit · IfcOpenShell", ifcopenshell.version)

## Teil A · Ein Modell mit Airtable-Links erkunden

`STR_mit_Airtable_Links.ifc` ist das BIMcollab-Tragwerksmodell. An 225 Bauteilen hängt ein Property Set `CPSet_Airtable`
mit Links zu zwei Airtable-Tabellen (Eigenschaften und Fertigung) – zusätzlich als `IfcDocumentReference`.

In [ ]:
modell = ifcopenshell.open(hole("STR_mit_Airtable_Links.ifc"))
zeilen = []
for el in modell.by_type("IfcBuildingElement"):
    ps = ue.get_psets(el).get("CPSet_Airtable")
    zeilen.append({"Klasse": el.is_a(), "mit Link": ps is not None})
df = pd.DataFrame(zeilen)
print(len(df), "Bauteile")
df.groupby("Klasse")["mit Link"].agg(["count", "sum"]).rename(columns={"count": "Bauteile", "sum": "mit Airtable-Link"})

In [ ]:
#@title Ein Bauteil im Detail
el = next(e for e in modell.by_type("IfcSlab") if "CPSet_Airtable" in ue.get_psets(e))
print(el.is_a(), el.GlobalId, el.Name)
for k, v in ue.get_psets(el)["CPSet_Airtable"].items():
    if k != "id": print(f"  {k}: {v}")
print("Dokument-Referenzen:")
for rel in el.HasAssociations:
    if rel.is_a("IfcRelAssociatesDocument"):
        d = rel.RelatingDocument
        print(f"  {d.Name}: {d.Location}")

**Probiert einen Link aus dem Modell im Browser.** Ihr landet auf einer Login-Seite: die Links zeigen direkt in die Basis,
und die ist nicht öffentlich.

👉 Das ist kein technischer Fehler, sondern eine **Informationsmanagement-Frage**: Wer darf die Daten sehen, wer pflegt sie,
was passiert, wenn die Basis gelöscht wird? Genau das gehört in den BEP.

**Die Daten selbst ansehen:** [Airtable-Basis „BFH IFC Demo“ (Lesezugriff, ohne Konto)](https://airtable.com/appNGbI0mNP8RhZzZ/shrhbra4Qz7Lc4fnS).
Sucht dort die GlobalId von oben. **Bearbeiten:** oben auf **Copy base** klicken (gratis Airtable-Konto) – dann gehört euch eine Kopie.

**Selbst ansehen:** `modelle/STR_mit_Airtable_Links.ifc` herunterladen und in [ifclite.com](https://www.ifclite.com) öffnen →
Bauteil anklicken → Eigenschaften → `CPSet_Airtable`.

## Teil B · Eigene Tabelle aus einem IFC

Wir machen aus dem Modell eine Tabelle mit **GlobalId** als Schlüssel. Die Spalten `Status` und `Lieferant` sind leer –
die füllt ihr in Airtable oder Google Sheets.

In [ ]:
#@title Tabelle erzeugen
klassen = ["IfcBeam", "IfcColumn", "IfcSlab", "IfcWall", "IfcFooting"]   # IfcWall schliesst IfcWallStandardCase ein
zeilen = []
for k in klassen:
    for el in modell.by_type(k):
        geschoss = ue.get_container(el)
        brand = next((p.get("FireRating") for n, p in ue.get_psets(el).items() if n.endswith("Common")), None)
        zeilen.append({"GlobalId": el.GlobalId, "Klasse": el.is_a(), "Name": el.Name,
                       "Geschoss": geschoss.Name if geschoss else None, "FireRating": brand,
                       "Status": "", "Lieferant": ""})
tabelle = pd.DataFrame(zeilen)
tabelle.to_csv("elemente.csv", index=False)
print(len(tabelle), "Zeilen → elemente.csv")
tabelle.head(8)

In [ ]:
#@title elemente.csv herunterladen
if files: files.download("elemente.csv")
else: print("Lokal: Datei liegt im aktuellen Ordner.")

## Teil C · In Airtable (oder Google Sheets) bearbeiten

**Airtable** (Konto gratis, airtable.com):
1. *Create* → *Import* → *CSV file* → `elemente.csv` hochladen.
2. Feld hinzufügen: Typ **Formula**, Name `Record ID`, Formel `RECORD_ID()`.
3. Bei einigen Bauteilen `Status` (z. B. *bestellt*, *geliefert*) und `Lieferant` eintragen.
4. Ansicht → *…* → **Download CSV**.
5. Die Adresse der Tabelle aus dem Browser kopieren: `https://airtable.com/app…/tbl…`

**Google Sheets:** *Datei → Importieren* → bearbeiten → *Datei → Herunterladen → CSV*. Als Link dient die Sheet-Adresse.

**Abkürzung:** die Demo-Basis mit *Copy base* kopieren ([Link](https://airtable.com/appNGbI0mNP8RhZzZ/shrhbra4Qz7Lc4fnS)) statt selbst zu importieren.

**Ohne Konto:** Teil D läuft mit der Beispieldatei `airtable_export_beispiel.csv`.

## Teil D · Tabelle zurück ans Modell hängen und prüfen

In [ ]:
#@title Export laden
quelle = "Beispieldatei" #@param ["Beispieldatei", "Eigene CSV hochladen"]
tabellen_url = "https://airtable.com/appBEISPIEL/tblBEISPIEL" #@param {type:"string"}
if quelle == "Eigene CSV hochladen" and files:
    datei = list(files.upload())[0]
else:
    datei = hole("airtable_export_beispiel.csv")
export = pd.read_csv(datei, dtype=str).fillna("")
print(len(export), "Zeilen aus", datei)
export.head()

In [ ]:
#@title Passen Tabelle und Modell zusammen?  (Fehlerklasse „Mapping“)
im_modell = {e.GlobalId for e in modell.by_type("IfcRoot")}
in_tabelle = set(export["GlobalId"])
erwartet = set(tabelle["GlobalId"])
print("Zeilen ohne Bauteil im Modell:", sorted(in_tabelle - im_modell))
print("Bauteile ohne Zeile in der Tabelle:", sorted(erwartet - in_tabelle))
doppelt = export["GlobalId"][export["GlobalId"].duplicated()].tolist()
print("Doppelte GlobalIds in der Tabelle:", doppelt)

In [ ]:
#@title Links ins IFC schreiben und speichern
import ifcopenshell.api
neu = ifcopenshell.open(hole("STR_mit_Airtable_Links.ifc"))
geschrieben = 0
for _, z in export.iterrows():
    try:
        el = neu.by_guid(z["GlobalId"])
    except RuntimeError:
        continue                                   # Zeile ohne Bauteil: oben schon gemeldet
    rec = z.get("Record ID", "")
    werte = {"Tabelle_URL": f"{tabellen_url}/{rec}" if rec else tabellen_url,
             "Record_ID": rec, "Status": z.get("Status", ""), "Lieferant": z.get("Lieferant", "")}
    ps = ifcopenshell.api.run("pset.add_pset", neu, product=el, name="CPSet_Kurs_Tabelle")
    ifcopenshell.api.run("pset.edit_pset", neu, pset=ps, properties={k: v for k, v in werte.items() if v})
    geschrieben += 1
neu.write("STR_mit_eigener_Tabelle.ifc")
print(geschrieben, "Bauteile verknüpft → STR_mit_eigener_Tabelle.ifc")
if files: files.download("STR_mit_eigener_Tabelle.ifc")

Öffnet `STR_mit_eigener_Tabelle.ifc` in [ifclite.com](https://www.ifclite.com) und klickt ein Bauteil an: `CPSet_Kurs_Tabelle`.

## Fragen für euer Projekt
- Welche eurer Daten gehören **ins** IFC, welche in eine Tabelle? Wer pflegt sie?
- Was passiert mit den Links, wenn das Modell neu exportiert wird und sich GlobalIds ändern?
- Wer hat Zugriff auf die Tabelle – und steht das im BEP?
- Euer Prüfergebnis (GlobalId, Ist, Regel, Ergebnis) ist selbst so eine Tabelle: Wer bekommt sie, in welchem Werkzeug?